# 查看当前GPU的显存信息

In [2]:
import pycuda.driver as cuda
cuda.init()

device = cuda.Device(0)  # 假设你只使用一个设备，选择设备 0
print("Device name:", device.name())
print("Total memory:", device.total_memory() // (1024 ** 2), "MB")
print("Compute capability:", device.compute_capability())

# 如果想获得更多关于设备的详细信息
print("Multi-processor count:", device.get_attribute(cuda.device_attribute.MULTIPROCESSOR_COUNT))
print("Clock rate:", device.get_attribute(cuda.device_attribute.CLOCK_RATE))  # 设备的时钟频率


Device name: NVIDIA A100-PCIE-40GB
Total memory: 40337 MB
Compute capability: (8, 0)
Multi-processor count: 108
Clock rate: 1410000


In [4]:
import pynvml

# 初始化NVML
pynvml.nvmlInit()

# 获取设备数量
device_count = pynvml.nvmlDeviceGetCount()

# 选择第一张 GPU
device = pynvml.nvmlDeviceGetHandleByIndex(0)

# 获取设备的名称
device_name = pynvml.nvmlDeviceGetName(device)
print(f"Device: {device_name}")

# 获取设备的各级缓存信息
# NVML API 不直接返回 L1 cache 大小，但可以通过该 API 获取设备的详细硬件配置
# 检索设备的详细信息
attributes = pynvml.nvmlDeviceGetAttributes(device)
for key, value in attributes.items():
    print(f"{key}: {value}")

# 结束NVML会话
pynvml.nvmlShutdown()


Device: NVIDIA A100-PCIE-40GB


NVMLError_NotSupported: Not Supported

In [8]:
import torch

# 假设 key_states 是一个 tensor
key_states = torch.randn(20, 40, 164, 128).cuda()  # Example tensor on GPU

# 获取元素个数
num_elements = key_states.numel()

# 获取每个元素的字节大小
element_size = key_states.element_size()

# 计算总内存占用（以字节为单位）
memory_size_bytes = num_elements * element_size

# 打印内存占用大小
print(f"key_states memory size: {memory_size_bytes / (1024 ** 2)} MB")


key_states memory size: 64.0625 MB


In [9]:
from transformers import AutoTokenizer, AutoModel
import torch  # 确保可以管理 GPU 资源

# 模型路径
model_name = "../Models/Llama-2-7b-chat-hf"

# 初始化分词器
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # 设置填充标记为 eos 标记

# 加载模型并移动到 GPU
model = AutoModel.from_pretrained(model_name).cuda(3)

print("Model",model)
# 用户输入
user_input = "What is the capital of China?"

# 编码输入（移除 token_type_ids）
encoded_input = tokenizer(user_input, return_tensors="pt")
if "token_type_ids" in encoded_input:  # 如果存在 token_type_ids，则删除
    del encoded_input["token_type_ids"]

# 将输入张量移动到 GPU
encoded_input = {key: value.cuda(3) for key, value in encoded_input.items()}

print("Encoded Input:", encoded_input)

# **获取模型输出**：前向传播计算 token 嵌入
outputs = model(**encoded_input)

# **查看每个 token 的嵌入向量**
token_embeddings = outputs.last_hidden_state  # (batch_size, sequence_length, embedding_dim)

# **查看嵌入形状**
print("Embedding Shape:", token_embeddings.shape)  # (1, sequence_length, embedding_dim)

# **逐个 token 查看嵌入向量**
for i, token in enumerate(tokenizer.tokenize(user_input)):
    print(f"Token: {token}")
    print(f"Embedding (size {token_embeddings[0, i].shape[0]}): {token_embeddings[0, i].detach().cpu().numpy()}\n")

# **清理 GPU 缓存**
torch.cuda.empty_cache()

# **删除模型和分词器**
del model
del tokenizer


Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.66s/it]
Some weights of the model checkpoint at ../Models/Llama-2-7b-chat-hf were not used when initializing LlamaModel: ['lm_head.weight']
- This IS expected if you are initializing LlamaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing LlamaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Model LlamaModel(
  (embed_tokens): Embedding(32000, 4096, padding_idx=0)
  (layers): ModuleList(
    (0-31): 32 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (rotary_emb): LlamaRotaryEmbedding()
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
        (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
        (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): LlamaRMSNorm()
      (post_attention_layernorm): LlamaRMSNorm()
    )
  )
  (norm): LlamaRMSNorm()
)
Encoded Input: {'input_ids': tensor([[    1,  1724,   338,

In [6]:
# **清理 GPU 缓存**
torch.cuda.empty_cache()

# **删除模型和分词器**
del model
del tokenizer